# NCAA March Madness Meta-Stacking (v5 Optimized)

This version includes optimized, vectorized prediction for much faster execution.

In [18]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
from sklearn.preprocessing import StandardScaler
import os

DATA_PATH = './march-machine-learning-mania-2026/'
team_features = pd.read_csv('all_team_features_v3.csv')
m_tourney = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))
tourney_results = pd.concat([m_tourney, w_tourney])

print("Data loaded.")

Data loaded.


In [19]:
def prepare_training_data_v5(results_df, features_df):
    training_rows = []
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    cols = ['SeedInt', 'OffEff', 'DefEff', 'Elo', 'AvgRank']
    
    for i, row in results_df.iterrows():
        s, w, l = row['Season'], row['WTeamID'], row['LTeamID']
        try:
            wf, lf = feat_dict[(s, w)], feat_dict[(s, l)]
            training_rows.append({**{f'{c}Diff': wf[c] - lf[c] for c in cols}, 'label': 1})
            training_rows.append({**{f'{c}Diff': lf[c] - wf[c] for c in cols}, 'label': 0})
        except KeyError: continue
    return pd.DataFrame(training_rows)

train_df = prepare_training_data_v5(tourney_results, team_features)
X = train_df.drop('label', axis=1)
y = train_df['label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Training data scaled and ready.")

Training data scaled and ready.


## 1. OOF Stacking & Meta-Model Training

In [20]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_xgb, oof_lgb, oof_nn = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X))

for tr_idx, val_idx in kf.split(X):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    X_tr_s, X_val_s = X_scaled[tr_idx], X_scaled[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    
    m_xgb = xgb.XGBClassifier(eval_metric='logloss', max_depth=4, learning_rate=0.05, n_estimators=150)
    m_xgb.fit(X_tr, y_tr)
    oof_xgb[val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    
    m_lgb = lgb.LGBMClassifier(verbose=-1, num_leaves=16, learning_rate=0.05, n_estimators=150)
    m_lgb.fit(X_tr, y_tr)
    oof_lgb[val_idx] = m_lgb.predict_proba(X_val)[:, 1]
    
    m_nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
    m_nn.fit(X_tr_s, y_tr)
    oof_nn[val_idx] = m_nn.predict_proba(X_val_s)[:, 1]

X_meta = pd.DataFrame({'xgb': oof_xgb, 'lgb': oof_lgb, 'nn': oof_nn})
meta_model = LogisticRegression()
meta_model.fit(X_meta, y)
print(f"Meta-learner trained. Ensemble Log Loss: {log_loss(y, meta_model.predict_proba(X_meta)[:, 1]):.4f}")

Meta-learner trained. Ensemble Log Loss: 0.5184


## 2. Optimized Batch Prediction

Instead of predicting row-by-row, we prepare all features at once and predict in batches. This is ~100x faster.

In [21]:
# Pre-train final base models
f_xgb = xgb.XGBClassifier(eval_metric='logloss', max_depth=4, learning_rate=0.05, n_estimators=150).fit(X, y)
f_lgb = lgb.LGBMClassifier(verbose=-1, num_leaves=16, learning_rate=0.05, n_estimators=150).fit(X, y)
f_nn = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42).fit(X_scaled, y)

submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))
feat_dict = team_features.set_index(['Season', 'TeamID']).to_dict('index')
cols = ['SeedInt', 'OffEff', 'DefEff', 'Elo', 'AvgRank']

print("Preparing batch test features...")
test_rows = []
missing_mask = []

for i, row in submission.iterrows():
    parts = row['ID'].split('_')
    s, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
    try:
        f1, f2 = feat_dict[(s, t1)], feat_dict[(s, t2)]
        test_rows.append({f'{c}Diff': f1[c] - f2[c] for c in cols})
        missing_mask.append(False)
    except KeyError:
        test_rows.append({f'{c}Diff': 0 for c in cols}) # Dummy data
        missing_mask.append(True)

X_test = pd.DataFrame(test_rows)
X_test_scaled = scaler.transform(X_test)

print("Running batch predictions...")
p_xgb = f_xgb.predict_proba(X_test)[:, 1]
p_lgb = f_lgb.predict_proba(X_test)[:, 1]
p_nn = f_nn.predict_proba(X_test_scaled)[:, 1]

X_test_meta = pd.DataFrame({'xgb': p_xgb, 'lgb': p_lgb, 'nn': p_nn})
final_preds = meta_model.predict_proba(X_test_meta)[:, 1]

# Apply clipping and handle missing data
final_preds = np.clip(final_preds, 0.02, 0.98)
final_preds[missing_mask] = 0.5

submission['Pred'] = final_preds
submission.to_csv('submission_v5.csv', index=False)
print("Optimized Submission v5 complete!")

Preparing batch test features...
Running batch predictions...
Optimized Submission v5 complete!
